# M25 lab solution: Verified answers

**Brief.** Add chain-of-verification to your RAG answer step and measure how many unsupported claims it removes on the golden set.

The solution answers the 25 golden questions that have evidence, twice: once with the plain answer step, once with chain-of-verification (plan checks, answer each check against the passages without the draft, revise). A judge prompt, printed below, splits each answer into claims and marks each supported or not.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## The Part V system, rebuilt inline

This lab does not depend on another lab's output, so the cell below rebuilds the minimal Part V system: the handbook loaded with `lab_kit`, the baseline chunker (500 characters, 50 overlap), an in-memory vector store over `get_embeddings()`, and an answer prompt that cites passages by number. Later cells reuse `retriever`, `format_docs` and `rag_answer`.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

handbook = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
embeddings = get_embeddings()
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook. Use only the numbered passages. "
               "Cite each claim with its passage number in square brackets, like [2]. "
               "If the passages do not contain the answer, reply: The handbook does not say."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(docs, 1))


def rag_answer(question, docs=None):
    """Retrieve (unless docs are given) and answer with citations; returns (answer, docs)."""
    docs = retriever.invoke(question) if docs is None else docs
    return answer_chain.invoke({"context": format_docs(docs), "question": question}), docs


print(len(chunks), "chunks;", len(golden), "golden questions")

## 1. Baseline answers

The plain answer step on the 25 answerable questions. The retrieved passages are kept, so the verified answers and the judge see the same evidence.

In [ ]:
answerable = [g for g in golden if g["evidence_spans"]]
baseline = {}
for g in answerable:
    text, docs = rag_answer(g["question"])
    baseline[g["id"]] = {"answer": text, "docs": docs}
first = answerable[0]["id"]
print(len(baseline), "baseline answers. First one:\n", baseline[first]["answer"])

## 2. Chain-of-verification in the answer step

The four steps of `04a_chain_of_verification.ipynb`: draft (the baseline answer), plan one verification question per claim, answer each verification question against the passages in a separate call that never sees the draft, then revise the draft to keep only what the checks support.

In [ ]:
from pydantic import BaseModel, Field


class VerificationPlan(BaseModel):
    questions: list[str] = Field(description="Short fact-checking questions, one per factual claim in the draft")


PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "List one short verification question for each factual claim in the draft answer. "
               "Ask about the fact itself (a number, a rule, a name), not about the draft."),
    ("human", "Question: {question}\n\nDraft answer:\n{draft}"),
])
VERIFY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer the verification question from the numbered passages only, in one sentence with a citation like [2]. "
               "If the passages do not answer it, reply exactly: NOT IN PASSAGES."),
    ("human", "Passages:\n{context}\n\nVerification question: {vq}"),
])
REVISE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the draft so that it keeps only the claims the verification answers support. "
               "Drop or correct any claim whose check came back NOT IN PASSAGES or disagrees with the draft. "
               "Keep the citations. If nothing is supported, reply: The handbook does not say."),
    ("human", "Question: {question}\n\nDraft answer:\n{draft}\n\nVerification:\n{checks}"),
])
planner = PLAN_PROMPT | llm.with_structured_output(VerificationPlan)
verifier = VERIFY_PROMPT | llm | StrOutputParser()
reviser = REVISE_PROMPT | llm | StrOutputParser()


def cove(question, draft, docs):
    context = format_docs(docs)
    plan = planner.invoke({"question": question, "draft": draft}).questions
    checks = [(vq, verifier.invoke({"context": context, "vq": vq})) for vq in plan]  # never sees the draft
    revised = reviser.invoke({"question": question, "draft": draft,
                              "checks": "\n".join(f"Q: {q}\nA: {a}" for q, a in checks)})
    return revised, checks


verified = {}
for g in answerable:
    b = baseline[g["id"]]
    revised, checks = cove(g["question"], b["answer"], b["docs"])
    verified[g["id"]] = {"answer": revised, "checks": checks}

print("Checks for", first)
for q, a in verified[first]["checks"]:
    print(" -", q, "->", a)
print("\nRevised answer:\n", verified[first]["answer"])

## 3. The judge

The judge splits an answer into factual claims and marks each one supported only when the retrieved passages state it. Both answer steps are judged against the same passages.

In [ ]:
class Claim(BaseModel):
    claim: str = Field(description="One factual claim made by the answer")
    supported: bool = Field(description="True only if the passages state this claim")


class Judgement(BaseModel):
    claims: list[Claim]


JUDGE_SYSTEM = ("You check answers against source passages. Split the answer into its factual claims "
                "(numbers, rules, names, conditions). For each claim, set supported to true only if the passages state it. "
                "A claim that goes beyond the passages, or contradicts them, is unsupported. "
                "A statement that the handbook does not say something is not a claim.")
JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", JUDGE_SYSTEM),
    ("human", "Passages:\n{context}\n\nQuestion: {question}\n\nAnswer:\n{answer}"),
])
judge = JUDGE_PROMPT | llm.with_structured_output(Judgement)
print("Judge prompt (system message):\n" + JUDGE_SYSTEM)


def judged(question, answer, docs):
    claims = judge.invoke({"context": format_docs(docs), "question": question, "answer": answer}).claims
    return len(claims), sum(not c.supported for c in claims), [c.claim for c in claims if not c.supported]


rows = []
for g in answerable:
    docs = baseline[g["id"]]["docs"]
    n_before, u_before, flagged = judged(g["question"], baseline[g["id"]]["answer"], docs)
    n_after, u_after, _ = judged(g["question"], verified[g["id"]]["answer"], docs)
    rows.append({"id": g["id"], "type": g["type"], "claims_before": n_before, "unsupported_before": u_before,
                 "claims_after": n_after, "unsupported_after": u_after, "flagged_before": "; ".join(flagged)})
per_question = pd.DataFrame(rows)
print(per_question.drop(columns="flagged_before").to_string(index=False))

## 4. Results

In [ ]:
summary = pd.DataFrame({
    "answer step": ["baseline", "chain-of-verification"],
    "questions": [len(per_question)] * 2,
    "claims": [per_question.claims_before.sum(), per_question.claims_after.sum()],
    "unsupported claims": [per_question.unsupported_before.sum(), per_question.unsupported_after.sum()],
    "answers with an unsupported claim": [(per_question.unsupported_before > 0).sum(), (per_question.unsupported_after > 0).sum()],
})
print(summary.to_string(index=False))
print("\nClaims the judge flagged in baseline answers:")
for _, r in per_question[per_question.unsupported_before > 0].iterrows():
    print(f"  {r.id}: {r.flagged_before}")

## What this shows

- The unsupported-claims row is the number the brief asks for. Read it beside the claims row: a revision that removes claims can lower the unsupported count by saying less, so check that supported claims did not drop with them.
- Chain-of-verification adds one planning call, one call per verification question and one revision call to every answer. The per-question table shows where that cost bought a change.
- The judge is a model too. Read the flagged claims above before trusting the totals; a claim marked unsupported because the judge split a sentence oddly is a judge error, not an answer error.